# Application - knowledge cards, store, ask-your-predecessor, handover packs

1. **Extract** cards from knowledge-bearing predictions (Stage 14). Public sources: Claude (costs money); own logs: local extractive backend, always.
2. **Store** them: SQLite + vectors + graph.
3. **Ask**, **search**, build **handover packs**. The same functions serve the REST API (`uvicorn krs.app.agent:app`) and the MCP server (`claude mcp add krs -- python -m krs.app.mcp_server`).

Settings: `configs/app.yaml`. Own-log cards never go to a cloud model.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[ml,app]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.app import extraction
from krs.config import get_paths
from krs import registry

version = registry.load()["version"]
preds = get_paths().predictions / f"{version}.jsonl"
cards_file = get_paths().store.parent / "cards.jsonl"
extraction.extract_cards(preds, cards_file)   # resumable; rejected cards in cards_rejected.jsonl

In [ ]:
from krs.app.storage import KnowledgeStore
from krs.io import read_jsonl
from krs.schema import KnowledgeCard

store = KnowledgeStore()
store.add(read_jsonl(cards_file, KnowledgeCard))
store.stats()

## Ask your predecessor

In [ ]:
from krs.app.agent import ask

r = ask(store, "Why did we choose a queue for order emails?")   # add filters, e.g. {"project_id": "orders"}
print(r["answer"])
r["citations"], r["unknown_citations"], r["note"]

In [ ]:
from krs.app.retrieval import search

[(h.score, h.via, h.card.title) for h in search(store, "how do we deploy", k=5, expand=True)]

## Handover pack (per project or per person)

In [ ]:
from IPython.display import Markdown
from krs.app.handover import build_pack

project = next(c.project_id for c in store.all() if c.project_id)
Markdown(build_pack(store, project_id=project))

## GDPR erasure

`store.delete_person("p01")` removes every card of that person from the database, the vectors and the graph - no retraining needed.